# World Commodities (Yahoo Finance)

## Setup

In [1]:
# Standard library
from pathlib import Path
from typing import cast

import mgplot as mg

# Third party
import pandas as pd
import certifi
import requests
import xml.etree.ElementTree as ET
import yfinance as yf
from curl_cffi import requests as cffi_requests

# Local
from abs_structured_capture import ReqsTuple, get_abs_data as load_abs_structured

In [2]:
# Pandas display
pd.options.display.max_rows = 999999

# Chart output directory
CHART_DIR = "./CHARTS/Commodities/"
mg.set_chart_dir(CHART_DIR)
mg.clear_chart_dir()

# Plotting control
SHOW = False
SOURCE_YAHOO = "Source: Yahoo Finance"

# Date-range starts
SHORT_START = "2025-01-01"   # history start for oil/gas/refined-product/crack charts; set to "2026-01-01" for the short post-disruption window
LONG_START = "2024-03-20"    # ~2-year window for metals, grains, softs, indices

# Shared event annotation
HORMUZ_CLOSURE = {
    "x": pd.Period("2026-02-28", freq="D"),
    "color": "grey",
    "linestyle": "--",
    "linewidth": 1,
    "label": "Hormuz closure",
}

# Oilprice.com source cache
CACHE_DIR = Path("./CACHE/YAHOO_oilprice/")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OILPRICE_BLENDS = {
    "WTI (US)": {"blend_id": 45, "cache": "wti.csv"},
    "Brent (Europe)": {"blend_id": 46, "cache": "brent.csv"},
    "Oman (Middle East)": {"blend_id": 48, "cache": "dme_oman.csv"},
    "Dubai (Middle East)": {"blend_id": 144, "cache": "dubai.csv"},
}

# EIA daily spot price source (no API key required)
EIA_CACHE_DIR = Path("./CACHE/EIA_spot/")
EIA_CACHE_DIR.mkdir(parents=True, exist_ok=True)
EIA_SPOT = {
    "WTI (Cushing)": {"url": "https://www.eia.gov/dnav/pet/hist_xls/RWTCd.xls", "cache": "wti.csv"},
    "Brent (Europe)": {"url": "https://www.eia.gov/dnav/pet/hist_xls/RBRTEd.xls", "cache": "brent.csv"},
}

# EIA refined product spot prices (USD/gallon; multiply by US_GAL_PER_BBL for USD/bbl)
EIA_GULF_JET_URL = "https://www.eia.gov/dnav/pet/hist_xls/eer_epjk_pf4_rgc_dpgd.xls"


# OPEC Reference Basket (Cloudflare-protected; need TLS-fingerprint client)
OPEC_URL = "https://www.opec.org/basket/basketDayArchives.xml"
OPEC_CACHE_DIR = Path("./CACHE/OPEC_basket/")
OPEC_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# CME dated-futures codes used to build forward-curve symbols
MONTH_CODES = "FGHJKMNQUVXZ"
N_FORWARD_MONTHS = 18

# CME settlements API for Singapore refined-product forward curves.
# Yahoo only exposes the front-month for SGB (gasoil) and N1B (mogas 92);
# CME publishes daily settles for all dated months on their public web API.
# The settlements page is Akamai-protected, so we need curl_cffi TLS impersonation.
CME_SETTLEMENTS_BASE = "https://www.cmegroup.com/CmeWS/mvc/Settlements/Futures"
CME_REFERER = (
    "https://www.cmegroup.com/markets/energy/refined-products/"
    "singapore-gasoil-swap-futures.settlements.html"
)
CME_SINGAPORE_PRODUCTS = {
    "Sing. Gasoil 10ppm": 5033,   # Singapore Gasoil (Platts) Swap Futures (USD/bbl)
    "Sing. Mogas 92": 4544,       # Singapore Mogas 92 Unleaded (Platts) Swap Futures (USD/bbl)
}

# Natural gas unit conversion (TTF quoted in EUR per MWh)
MMBTU_PER_MWH = 3.412

# Refined product unit conversion (EIA jet fuel quoted in USD per US gallon)
US_GAL_PER_BBL = 42

# Forward-curve "return to normal" reference lines.
# Per series: a dashed, finer line in the series colour at the 2025 mean
# front-month price x a long-run-volatility tolerance. Means are hard-coded
# (continuous front-month daily close, calendar 2025) and quoted in USD/bbl
# to match the curve charts.
FWD_CURVE_COLORS = ["darkblue", "darkorange"]
NORMAL_TOLERANCE = 1.2  # uplift on the 2025 mean for long-run volatility
NORMAL_2025_MEAN = {     # USD per barrel
    "WTI (US)": 64.74,            # CL=F
    "Brent (Europe)": 68.10,      # BZ=F
    "Sing. Gasoil 10ppm": 87.40,  # SGB=F
    "Sing. Mogas 92": 78.58,      # N1B=F (USD/bbl)
}
CONTANGO_THRESHOLD_PCT = 0.01  # forward-average uplift over the trough that flags a sustained flip into contango

## Helper functions

In [3]:
def fetch_yahoo_close(ticker: str, start: str) -> pd.Series:
    """Fetch daily closing prices for a Yahoo ticker as a PeriodIndex series."""
    raw = yf.download(ticker, start=start, auto_adjust=True, progress=False)
    if raw is None or len(raw) == 0:
        return pd.Series(dtype=float)
    s = raw["Close"].squeeze().dropna()
    s.index = cast("pd.DatetimeIndex", s.index).to_period("D")
    return s


def fetch_yahoo_frame(tickers: dict[str, str], start: str) -> pd.DataFrame:
    """Fetch closing prices for multiple tickers into a single DataFrame."""
    frames = {label: fetch_yahoo_close(ticker, start) for ticker, label in tickers.items()}
    return pd.DataFrame(frames)


def summarise(df: pd.DataFrame, label: str) -> None:
    """Print date range and min/max summary for each column."""
    valid = df.dropna(how="all")
    print(f"{label}: {valid.index[0]} to {valid.index[-1]}")
    for col in df.columns:
        s = df[col].dropna()
        if len(s):
            print(f"  {col:22s}  n={len(s):3d}  min={s.min():.2f}  max={s.max():.2f}")


def latest_date_str(df: pd.DataFrame) -> str:
    """Return the last non-NaN date as a formatted string."""
    return df.dropna(how="all").index[-1].strftime("%-d-%b-%Y")

In [4]:
def plot_frame(
    df: pd.DataFrame,
    *,
    title: str,
    ylabel: str,
    lfooter: str,
    rfooter: str = SOURCE_YAHOO,
    axvline: dict | None = None,
) -> None:
    """Plot a multi-series DataFrame with standard chart formatting."""
    mg.line_plot_finalise(
        df,
        title=title,
        ylabel=ylabel,
        xlabel=None,
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        axvline=axvline,
        lfooter=lfooter,
        rfooter=rfooter,
        show=SHOW,
    )


def plot_single(
    series: pd.Series, *, name: str, ylabel: str, is_futures: bool = True
) -> None:
    """Plot a single commodity series with standard formatting."""
    data_to = series.index[-1].strftime("%-d-%b-%Y")
    if is_futures:
        title = f"{name} Futures Price"
        footer = f"Front-month futures. Data to {data_to}."
    else:
        title = name
        footer = f"Daily close. Data to {data_to}."
    mg.line_plot_finalise(
        series,
        title=title,
        ylabel=ylabel,
        xlabel=None,
        annotate=True,
        lfooter=footer,
        rfooter=SOURCE_YAHOO,
        show=SHOW,
    )


def chart_single_tickers(
    tickers: list[tuple[str, str, str]], start: str, is_futures: bool = True
) -> None:
    """Fetch and chart each (ticker, name, ylabel) tuple independently."""
    for ticker, name, ylabel in tickers:
        s = fetch_yahoo_close(ticker, start)
        if len(s) == 0:
            print(f"{name} ({ticker}): no data, skipping")
            continue
        print(f"{name}: {s.index[0]} to {s.index[-1]}  min={s.min():.2f}  max={s.max():.2f}")
        plot_single(s, name=name, ylabel=ylabel, is_futures=is_futures)

In [5]:
def _contract_months(n_months: int) -> list[pd.Period]:
    start = pd.Period(pd.Timestamp.today(), freq="M") + 1
    return [start + i for i in range(n_months)]


def _contract_symbol(root: str, period: pd.Period) -> str:
    code = MONTH_CODES[period.month - 1]
    yy = period.year % 100
    return f"{root}{code}{yy:02d}.NYM"


def fetch_forward_curve(root: str, n_months: int) -> pd.DataFrame:
    """Fetch latest close + trade date for each dated contract."""
    records = {}
    for m in _contract_months(n_months):
        sym = _contract_symbol(root, m)
        try:
            hist = yf.Ticker(sym).history(period="5d")
        except (KeyError, ValueError, OSError):
            continue
        if hist is None or len(hist) == 0:
            continue
        records[m] = {
            "price": float(hist["Close"].iloc[-1]),
            "date": hist.index[-1].tz_localize(None).normalize(),
        }
    return pd.DataFrame.from_dict(records, orient="index").sort_index()


def build_forward_curves(n_months: int) -> tuple[pd.DataFrame, pd.Timestamp]:
    """Fetch WTI and Brent forward curves and return prices with trade date."""
    wti = fetch_forward_curve("CL", n_months)
    brent = fetch_forward_curve("BZ", n_months)
    prices = pd.DataFrame({
        "WTI (US)": wti["price"],
        "Brent (Europe)": brent["price"],
    })
    prices.index = pd.PeriodIndex(prices.index, freq="M")
    latest = max(wti["date"].max(), brent["date"].max())
    return prices, latest


def _fetch_cme_curve(product_id: int) -> tuple[pd.Series, pd.Timestamp]:
    """Fetch a CME settlements curve for one product, return (series, trade date).

    Hits the public settlements JSON the cmegroup.com page itself uses.
    Akamai bot protection serves the API too, so we reuse the same curl_cffi
    chrome120-impersonating session pattern as the OPEC fetch.
    """
    sess = cffi_requests.Session(impersonate="chrome120")
    sess.get(CME_REFERER, timeout=30, verify=certifi.where())
    hdrs = {"Referer": CME_REFERER, "Accept": "application/json"}

    td_resp = sess.get(
        f"{CME_SETTLEMENTS_BASE}/TradeDate/{product_id}",
        headers=hdrs, timeout=30, verify=certifi.where(),
    )
    td_resp.raise_for_status()
    trade_dates = td_resp.json()
    if not trade_dates:
        raise ValueError(f"CME product {product_id}: no trade dates available")
    # API returns list of [MM/DD/YYYY, reportType] sorted newest first
    trade_date = trade_dates[0][0]

    s_resp = sess.get(
        f"{CME_SETTLEMENTS_BASE}/Settlements/{product_id}/FUT",
        params={"strategy": "DEFAULT", "tradeDate": trade_date, "pageSize": 500},
        headers=hdrs, timeout=30, verify=certifi.where(),
    )
    s_resp.raise_for_status()
    payload = s_resp.json()

    records: dict[pd.Period, float] = {}
    for row in payload.get("settlements", []):
        month_str = row.get("month", "")
        settle_str = (row.get("settle") or "").replace(",", "")
        if settle_str in ("", "-"):
            continue
        try:
            # CME returns months as e.g. "APR 26". pd.Period("APR 26", "M")
            # silently parses to year 1, collapsing every year onto the same key.
            period = pd.Period(pd.to_datetime(month_str, format="%b %y"), freq="M")
            price = float(settle_str)
        except (ValueError, KeyError):
            continue
        records[period] = price

    series = pd.Series(records).sort_index()
    return series, pd.Timestamp(trade_date)


def build_singapore_curves() -> tuple[pd.DataFrame, pd.Timestamp]:
    """Fetch Singapore Gasoil + Mogas 92 forward curves from CME settlements."""
    cols: dict[str, pd.Series] = {}
    last_date: pd.Timestamp | None = None
    for label, pid in CME_SINGAPORE_PRODUCTS.items():
        series, td = _fetch_cme_curve(pid)
        cols[label] = series
        last_date = td if last_date is None else max(last_date, td)
    df = pd.DataFrame(cols).sort_index()
    assert last_date is not None
    return df, last_date


def plot_forward_curves(
    df: pd.DataFrame,
    as_of: pd.Timestamp,
    *,
    title: str,
    rfooter: str,
) -> None:
    """Chart a forward curve frame (each column = one contract series)."""
    as_of_str = as_of.strftime("%-d-%b-%Y")
    # Implied front-to-back decline, averaged across columns: this is what
    # the futures market is pricing in for "normalisation" of the disruption.
    pct_changes = []
    for col in df.columns:
        valid = df[col].dropna()
        if len(valid) >= 2:
            pct_changes.append((valid.iloc[-1] / valid.iloc[0] - 1) * 100)
    avg_pct = sum(pct_changes) / len(pct_changes) if pct_changes else 0.0
    back_period = df.dropna(how="all").index[-1]
    rheader = f"Market prices {avg_pct:+.0f}% by {back_period.strftime('%b %Y')}"

    # "Return to normal" reference per series: 2025 mean x tolerance, drawn
    # in the series colour but dashed and finer. Labelled so each line gets
    # its own legend entry (mgplot applies axhline before building the legend).
    ref_lines = [
        {
            "y": NORMAL_2025_MEAN[col] * NORMAL_TOLERANCE,
            "color": FWD_CURVE_COLORS[i % len(FWD_CURVE_COLORS)],
            "linestyle": "--",
            "linewidth": 0.9,
            "label": f"{col}: 2025 mean x{NORMAL_TOLERANCE:g}",
        }
        for i, col in enumerate(df.columns)
        if col in NORMAL_2025_MEAN
    ]

    # First flip into contango per series, defined as a sustained turn (not a
    # single-month blip): on a curve that begins in backwardation, the first
    # contract month where the next month settles higher AND the average of all
    # later months sits more than the threshold above it. Dash-dot vertical
    # line in the series colour (labelled for the legend).
    cross_lines = []
    threshold = CONTANGO_THRESHOLD_PCT / 100
    for i, col in enumerate(df.columns):
        valid = df[col].dropna()
        if len(valid) < 2 or valid.iloc[1] >= valid.iloc[0]:  # must start in backwardation
            continue
        cross_at = next(
            (
                valid.index[j]
                for j in range(len(valid) - 1)
                if valid.iloc[j + 1] > valid.iloc[j]
                and valid.iloc[j + 1:].mean() / valid.iloc[j] - 1 > threshold
            ),
            None,
        )
        if cross_at is None:
            continue
        cross_lines.append({
            "x": cross_at,
            "color": FWD_CURVE_COLORS[i % len(FWD_CURVE_COLORS)],
            "linestyle": "-.",
            "linewidth": 0.9,
            "label": f"{col}: backwardation to contango",
        })

    mg.line_plot_finalise(
        df,
        title=title,
        ylabel="USD per barrel",
        xlabel="Contract month",
        color=FWD_CURVE_COLORS[: df.shape[1]],
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        rounding=2,
        marker="o",
        markersize=4,
        rheader=rheader,
        axhline=ref_lines,
        axvline=cross_lines,
        lfooter=(
            f"Latest settle price for each dated contract (delivery within month). "
            f"As of {as_of_str}."
        ),
        rfooter=rfooter,
        show=SHOW,
    )

In [6]:
def _fetch_oilprice(blend_id: int, period: int) -> pd.DataFrame:
    """Fetch crude oil prices from Oilprice.com's public JSON endpoint."""
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36"
        ),
        "X-Requested-With": "XMLHttpRequest",
        "Referer": "https://oilprice.com/oil-price-charts/",
    }
    csrf = requests.get(
        "https://oilprice.com/ajax/csrf", headers=headers, timeout=15
    ).json()
    resp = requests.post(
        "https://oilprice.com/freewidgets/json_get_oilprices",
        headers=headers,
        data={"blend_id": blend_id, "period": period, csrf["name"]: csrf["hash"]},
        timeout=15,
    )
    resp.raise_for_status()
    df = pd.DataFrame(resp.json()["prices"])
    df["date"] = pd.to_datetime(df["time"], unit="s").dt.normalize()
    df["price"] = df["price"].astype(float)
    df = df.sort_values("date").reset_index(drop=True)
    # Drop stale artifact entries (timestamps far out of sequence)
    if len(df) > 1:
        df = df[df["date"] >= df["date"].iloc[-1] - pd.Timedelta(days=400)]
    return df[["date", "price"]]


def load_oilprice_blends() -> dict[str, pd.Series]:
    """Load all configured Oilprice.com blends, refreshing the on-disk cache."""
    result: dict[str, pd.Series] = {}
    for name, cfg in OILPRICE_BLENDS.items():
        cache_file = CACHE_DIR / cfg["cache"]
        if cache_file.exists():
            cached = pd.read_csv(cache_file, parse_dates=["date"])
            fresh = _fetch_oilprice(cfg["blend_id"], period=6)
            combined = pd.concat([cached, fresh]).drop_duplicates(subset="date", keep="last")
            df = combined.sort_values("date").reset_index(drop=True)
            print(f"{name}: updated cache ({len(cached)} -> {len(df)} rows)")
        else:
            df = _fetch_oilprice(cfg["blend_id"], period=5)
            print(f"{name}: initial cache ({len(df)} rows)")
        df.to_csv(cache_file, index=False)
        idx = pd.PeriodIndex(df["date"], freq="D")
        s = pd.Series(df["price"].values, index=idx, name=name)
        # Guard against duplicate dates in source data
        s = s[~s.index.duplicated(keep="last")]
        result[name] = s
    return result

In [7]:
def _fetch_eia_spot(url: str) -> pd.DataFrame:
    """Fetch an EIA daily spot-price Excel file and return a clean date/price frame."""
    df = pd.read_excel(url, sheet_name="Data 1", skiprows=2)
    df.columns = ["date", "price"]
    df = df.dropna()
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["price"] = df["price"].astype(float)
    return df.sort_values("date").reset_index(drop=True)


def load_eia_spot() -> dict[str, pd.Series]:
    """Load EIA daily spot prices for each configured benchmark, refreshing the cache."""
    result: dict[str, pd.Series] = {}
    for name, cfg in EIA_SPOT.items():
        df = _fetch_eia_spot(cfg["url"])
        df.to_csv(EIA_CACHE_DIR / cfg["cache"], index=False)
        print(f"{name}: {len(df)} rows  ({df['date'].min().date()} to {df['date'].max().date()})")
        idx = pd.PeriodIndex(df["date"], freq="D")
        s = pd.Series(df["price"].values, index=idx, name=name)
        result[name] = s[~s.index.duplicated(keep="last")]
    return result


def _fetch_opec_basket() -> pd.DataFrame:
    """Fetch OPEC Reference Basket daily archive via Cloudflare-bypassing client."""
    # opec.org sits behind Cloudflare bot protection. Plain `requests` returns 403,
    # cloudscraper also fails on the Turnstile JS challenge. curl_cffi with chrome120
    # TLS-fingerprint impersonation clears the gate without needing a JS engine.
    # certifi.where() avoids macOS "unable to get local issuer certificate" errors.
    r = cffi_requests.get(
        OPEC_URL, impersonate="chrome120", timeout=30, verify=certifi.where()
    )
    r.raise_for_status()
    root = ET.fromstring(r.text)
    ns = {"b": "http://tempuri.org/basketDayArchives.xsd"}
    records = [
        (e.get("data"), float(e.get("val")))
        for e in root.findall("b:BasketList", ns)
    ]
    df = pd.DataFrame(records, columns=["date", "price"])
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    return df.sort_values("date").reset_index(drop=True)


def load_opec_basket() -> pd.Series:
    """Load OPEC Reference Basket as a daily Series, refreshing the cache."""
    df = _fetch_opec_basket()
    df.to_csv(OPEC_CACHE_DIR / "opec_basket.csv", index=False)
    print(f"OPEC Basket: {len(df)} rows  ({df['date'].min().date()} to {df['date'].max().date()})")
    idx = pd.PeriodIndex(df["date"], freq="D")
    s = pd.Series(df["price"].values, index=idx, name="OPEC Basket")
    return s[~s.index.duplicated(keep="last")]

## Crude oil: physical spot benchmarks (EIA + OPEC)

In [8]:
def run_physical_spot() -> pd.DataFrame:
    """Fetch and chart EIA daily spot for WTI/Brent and the OPEC Reference Basket."""
    eia = load_eia_spot()
    orb = load_opec_basket()
    prices = pd.DataFrame({**eia, "OPEC Basket": orb}).sort_index()
    prices = prices.loc[prices.index >= pd.Period(SHORT_START, freq="D")]
    summarise(prices, "Physical spot")

    eia_last = prices["WTI (Cushing)"].dropna().index[-1].strftime("%-d-%b")
    orb_last = prices["OPEC Basket"].dropna().index[-1].strftime("%-d-%b")

    mg.line_plot_finalise(
        prices,
        title="Crude Oil Physical Spot: WTI, Brent and OPEC Basket",
        ylabel="USD per barrel",
        xlabel=None,
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        rounding=2,
        axvline=HORMUZ_CLOSURE,
        lfooter=f"Daily physical spot. EIA WTI/Brent to {eia_last}; OPEC Basket to {orb_last}.",
        rfooter="Source: U.S. EIA, OPEC Secretariat",
        show=SHOW,
    )
    return prices


physical_spot = run_physical_spot()

WTI (Cushing): 10163 rows  (1986-01-02 to 2026-05-18)


Brent (Europe): 9893 rows  (1987-05-20 to 2026-05-18)


OPEC Basket: 6033 rows  (2003-01-02 to 2026-05-25)
Physical spot: 2025-01-02 to 2026-05-25
  WTI (Cushing)           n=342  min=55.44  max=114.58
  Brent (Europe)          n=347  min=59.93  max=138.21
  OPEC Basket             n=359  min=58.39  max=146.05


## Crude oil: global benchmark futures

In [9]:
def run_crude_spot() -> pd.DataFrame:
    """Fetch and chart front-month crude oil futures from Oilprice.com."""
    spot_series = load_oilprice_blends()
    prices = pd.DataFrame(spot_series)
    prices = prices.sort_index()
    prices = prices.loc[prices.index >= pd.Period(SHORT_START, freq="D")]

    summarise(prices, "Crude futures (Oilprice.com)")

    # Per-series last dates for footer
    last_dates = {
        col: prices[col].dropna().index[-1].strftime("%-d-%b")
        for col in prices.columns
    }
    date_str = ", ".join(f"{col.split(' ')[0]} {d}" for col, d in last_dates.items())
    lfooter = f"Daily front-month futures (Dubai is a Platts assessment). Last: {date_str}."

    mg.line_plot_finalise(
        prices,
        title="Crude Oil Front-Month Futures: Global Benchmarks",
        ylabel="USD per barrel",
        xlabel=None,
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        rounding=2,
        axvline=HORMUZ_CLOSURE,
        lfooter=lfooter,
        rfooter="Source: Oilprice.com",
        show=SHOW,
    )
    return prices


spot_prices = run_crude_spot()

WTI (US): updated cache (344 -> 344 rows)


Brent (Europe): updated cache (344 -> 344 rows)


Oman (Middle East): updated cache (235 -> 235 rows)


Dubai (Middle East): updated cache (241 -> 241 rows)
Crude futures (Oilprice.com): 2025-03-21 to 2026-05-28
  WTI (US)                n=344  min=55.13  max=112.95
  Brent (Europe)          n=344  min=58.67  max=114.44
  Oman (Middle East)      n=235  min=58.36  max=162.72
  Dubai (Middle East)     n=241  min=13.55  max=137.82


## Crude oil: front-month futures

In [10]:
def run_crude_futures() -> pd.DataFrame:
    """Fetch and chart WTI vs Brent front-month futures."""
    df = fetch_yahoo_frame(
        {"CL=F": "WTI (US)", "BZ=F": "Brent (Europe)"},
        start=SHORT_START,
    )
    summarise(df, "Crude futures")
    mg.line_plot_finalise(
        df,
        title="Crude Oil Futures: WTI vs Brent",
        ylabel="USD per barrel",
        xlabel=None,
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        rounding=2,
        axvline=HORMUZ_CLOSURE,
        lfooter=f"Front-month futures. NYMEX (WTI) and ICE (Brent). Data to {latest_date_str(df)}.",
        rfooter=SOURCE_YAHOO,
        show=SHOW,
    )
    return df


wti_brent = run_crude_futures()

Crude futures: 2025-01-02 to 2026-05-27
  WTI (US)                n=352  min=55.27  max=112.95
  Brent (Europe)          n=352  min=58.92  max=118.35


## Crude oil: forward curves

In [11]:
def run_forward_curves() -> None:
    """Fetch and chart crude oil forward curves."""
    forward, as_of = build_forward_curves(N_FORWARD_MONTHS)
    print(f"As of {as_of.date()}")
    print(f"Contracts: {forward.index[0]} to {forward.index[-1]}")
    for col in forward.columns:
        valid = forward[col].dropna()
        print(f"  {col:20s}  n={len(valid)}  front={valid.iloc[0]:.2f}  back={valid.iloc[-1]:.2f}")
    plot_forward_curves(
        forward,
        as_of,
        title="Crude Oil Forward Curves: WTI and Brent",
        rfooter="Source: Yahoo Finance (NYMEX CL, ICE BZ)",
    )


run_forward_curves()

$CLM26.NYM: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")


$BZM26.NYM: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")


As of 2026-05-27
Contracts: 2026-07 to 2027-11
  WTI (US)              n=17  front=90.16  back=72.49
  Brent (Europe)        n=17  front=96.12  back=77.67


## Singapore refined products: forward curves

In [12]:
def run_singapore_forward_curves() -> None:
    """Fetch and chart Singapore Gasoil + Mogas 92 forward curves from CME settlements."""
    sing, as_of = build_singapore_curves()
    print(f"As of {as_of.date()}")
    print(f"Contracts: {sing.index[0]} to {sing.index[-1]}")
    for col in sing.columns:
        valid = sing[col].dropna()
        print(f"  {col:22s}  n={len(valid)}  front={valid.iloc[0]:.2f}  back={valid.iloc[-1]:.2f}")
    plot_forward_curves(
        sing,
        as_of,
        title="Singapore Refined Product Forward Curves: Gasoil and Petrol",
        rfooter="Source: CME Group (NYMEX SGB, N1B settlements)",
    )


run_singapore_forward_curves()

As of 2026-05-26
Contracts: 2026-05 to 2029-04
  Sing. Gasoil 10ppm      n=32  front=156.45  back=95.73
  Sing. Mogas 92          n=36  front=130.10  back=79.83


## Middle East crude differentials vs Brent

In [13]:
def run_crude_differentials(spot_prices: pd.DataFrame) -> None:
    """Plot Middle East crude differentials vs Brent.

    In normal times Dubai/Oman trade at a small discount to Brent. When
    Hormuz transits are constrained and ADCOP/Petroline bypass capacity is
    binding, Middle East crude that escapes the Gulf commands a scarcity
    premium. A widening Dubai-Brent or Oman-Brent spread is the cleanest
    physical-market signal of selective transit.
    """
    diffs = pd.DataFrame({
        "Oman minus Brent": spot_prices["Oman (Middle East)"] - spot_prices["Brent (Europe)"],
        "Dubai minus Brent": spot_prices["Dubai (Middle East)"] - spot_prices["Brent (Europe)"],
    }).dropna(how="all")

    summarise(diffs, "ME crude differentials")

    mg.line_plot_finalise(
        diffs,
        title="Middle East Crude Differentials vs Brent",
        ylabel="USD per barrel",
        xlabel=None,
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        rounding=2,
        axvline=HORMUZ_CLOSURE,
        axhspan={"ymin": -3, "ymax": 0, "color": "grey", "alpha": 0.2},
        y0=True,
        lfooter=(
            f"Grey band = normal range (Dubai/Oman trade ~$2 below Brent). "
            f"Data to {latest_date_str(diffs)}."
        ),
        rfooter="Source: Oilprice.com",
        show=SHOW,
    )


run_crude_differentials(spot_prices)

ME crude differentials: 2025-04-15 to 2026-05-26
  Oman minus Brent        n=220  min=-16.44  max=58.75
  Dubai minus Brent       n=225  min=-7.66  max=36.05


## Singapore refined products: gasoil and petrol

In [14]:
def run_singapore_products() -> None:
    """Fetch and chart Singapore Gasoil and Mogas 92 front-month futures."""
    gasoil = fetch_yahoo_close("SGB=F", start=SHORT_START)   # USD/bbl
    mogas = fetch_yahoo_close("N1B=F", start=SHORT_START)    # USD/bbl
    df = pd.DataFrame({
        "Gasoil 10ppm (diesel)": gasoil,
        "Mogas 92 (petrol)": mogas,
    })
    summarise(df, "Singapore products")
    mg.line_plot_finalise(
        df,
        title="Singapore Refined Product Futures: Gasoil and Petrol",
        ylabel="USD per barrel",
        xlabel=None,
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        rounding=2,
        axvline=HORMUZ_CLOSURE,
        rheader="Refined products (not crude)",
        lfooter=(
            f"NYMEX front-month futures on Platts Singapore Gasoil (diesel) "
            f"and Mogas 92 (petrol). Data to {latest_date_str(df)}."
        ),
        rfooter=SOURCE_YAHOO,
        show=SHOW,
    )


run_singapore_products()

Singapore products: 2025-01-02 to 2026-05-26
  Gasoil 10ppm (diesel)   n=350  min=75.95  max=232.57
  Mogas 92 (petrol)       n=350  min=69.81  max=142.95


## Refined product crack-spreads vs Brent

In [15]:
def _eia_jet_per_bbl() -> pd.Series:
    """Fetch EIA U.S. Gulf Coast Kerosene-Type Jet Fuel daily spot, USD/bbl."""
    df = _fetch_eia_spot(EIA_GULF_JET_URL)
    s = pd.Series(
        df["price"].values * US_GAL_PER_BBL,
        index=pd.PeriodIndex(df["date"], freq="D"),
        name="US Gulf Jet",
    )
    return s[~s.index.duplicated(keep="last")]


def run_product_crack_spreads() -> None:
    """Plot refined product crack-spreads vs Brent.

    The crack-spread = refined product price minus crude. When shipping
    disruption keeps product from market, crack-spreads widen even as
    crude softens. Plotting Singapore (gasoil, mogas) alongside US Gulf
    (jet) tests the regional dimension: if Hormuz is the binding shipping
    constraint, Singapore distillate should widen MORE than US Gulf.
    """
    brent = fetch_yahoo_close("BZ=F", start=SHORT_START)
    gasoil = fetch_yahoo_close("SGB=F", start=SHORT_START)         # USD/bbl
    mogas_bbl = fetch_yahoo_close("N1B=F", start=SHORT_START)      # USD/bbl
    gulf_jet = _eia_jet_per_bbl().loc[pd.Period(SHORT_START, "D"):]

    cracks: dict[str, pd.Series] = {
        "Sing. Gasoil 10ppm minus Brent": gasoil - brent,
        "Sing. Mogas 92 minus Brent": mogas_bbl - brent,
        "US Gulf Jet minus Brent": gulf_jet - brent,
    }

    # Probe Yahoo for a Singapore jet kerosene ticker. As of 2026-04 none of
    # the obvious NYMEX-style symbols return data; left here so the chart
    # picks one up automatically if Yahoo starts publishing it.
    jet_candidates = ("SJK=F", "S1K=F", "AKB=F", "JKE=F")
    for jet_ticker in jet_candidates:
        sing_jet = fetch_yahoo_close(jet_ticker, start=SHORT_START)
        if len(sing_jet) > 0:
            cracks[f"Sing. Jet ({jet_ticker}) minus Brent"] = sing_jet - brent
            print(f"Singapore jet kerosene ticker found: {jet_ticker} ({len(sing_jet)} rows)")
            break
    else:
        print(
            "No free Yahoo ticker for Singapore jet kerosene "
            f"(tried {', '.join(jet_candidates)}); using US Gulf Jet as regional proxy."
        )

    df = pd.DataFrame(cracks).dropna(how="all")
    summarise(df, "Refined product crack-spreads")

    mg.line_plot_finalise(
        df,
        title="Refined Product Crack-Spreads vs Brent",
        ylabel="USD per barrel",
        xlabel=None,
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        rounding=2,
        axvline=HORMUZ_CLOSURE,
        y0=True,
        lfooter=f"Sing.: NYMEX futures; US Gulf: EIA spot. Data to {latest_date_str(df)}.",
        rfooter="Source: Yahoo Finance, U.S. EIA",
        show=SHOW,
    )


run_product_crack_spreads()

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: SJK=F"}}}


$SJK=F: possibly delisted; no timezone found



1 Failed download:


['SJK=F']: possibly delisted; no timezone found


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: S1K=F"}}}


...

['AKB=F']: possibly delisted; no timezone found


$JKE=F: possibly delisted; no timezone found



1 Failed download:


['JKE=F']: possibly delisted; no timezone found


No free Yahoo ticker for Singapore jet kerosene (tried SJK=F, S1K=F, AKB=F, JKE=F); using US Gulf Jet as regional proxy.
Refined product crack-spreads: 2025-01-02 to 2026-05-26
  Sing. Gasoil 10ppm minus Brent  n=350  min=11.53  max=122.80
  Sing. Mogas 92 minus Brent  n=350  min=0.74  max=33.92
  US Gulf Jet minus Brent  n=342  min=13.29  max=74.88


## Broad commodity index

In [16]:
def run_bcom(start: str = "1990-12-31") -> None:
    """Chart Bloomberg Commodity Index over two horizons: ~2 year and full history."""

    bcom = fetch_yahoo_close("^BCOM", start=start)
    if len(bcom) == 0:
        print("^BCOM: no data")
        return
    print(f"^BCOM: {bcom.index[0]} to {bcom.index[-1]}  n={len(bcom)}  "
          f"min={bcom.min():.2f}  max={bcom.max():.2f}")

    plot_single(
        bcom.loc[pd.Period(LONG_START, freq="D"):],
        name="Bloomberg Commodity Index",
        ylabel="Index",
        is_futures=False,
    )
    plot_single(
        bcom,
        name="Bloomberg Commodity Index: Since 1991",
        ylabel="Index",
        is_futures=False,
    )


run_bcom()

^BCOM: 1991-01-02 to 2026-05-27  n=8888  min=59.48  max=237.96


In [17]:
def _above_threshold_runs(
    s: pd.Series, threshold: float, start: pd.Period | None = None
) -> list[tuple]:
    """Return contiguous (start, end) index runs where s > threshold."""
    if start is not None:
        s = s.loc[s.index.to_timestamp(how="end").normalize() >= start.to_timestamp()]
    above = (s > threshold).fillna(False)
    runs: list[tuple] = []
    run_start = None
    last = None
    for idx, val in above.items():
        if val and run_start is None:
            run_start = idx
        elif not val and run_start is not None:
            runs.append((run_start, last))
            run_start = None
        last = idx
    if run_start is not None:
        runs.append((run_start, last))
    return runs


def run_bcom_vs_cpi(start: str = "1991-12-31") -> None:
    """BCOM YoY (daily) and AU CPI YoY (quarterly observations on daily axis)."""
    # Fetch BCOM one year before `start` so the 252-trading-day YoY
    # series is already defined at the chart start.
    fetch_start = (pd.Timestamp(start) - pd.DateOffset(years=1)).strftime("%Y-%m-%d")
    bcom = fetch_yahoo_close("^BCOM", start=fetch_start)
    bcom_yoy = ((bcom / bcom.shift(252)) - 1) * 100
    bcom_yoy = bcom_yoy.dropna() / 8.0

    cpi_data = load_abs_structured({
        "AU CPI YoY": ReqsTuple(
            "6401.0", "64010Appendix1a",
            "All groups CPI, seasonally adjusted ;", "S", "",
            True, False, "",
        ),
    })
    au_cpi_yoy = cpi_data["AU CPI YoY"].series

    cpi_daily = pd.Series(
        au_cpi_yoy.values,
        index=pd.PeriodIndex(
            au_cpi_yoy.index.to_timestamp(how="end").normalize(), freq="D",
        ),
        name="AU CPI YoY (quarterly, SA)",
    )

    start_period = pd.Period(start, "D")
    df = pd.DataFrame({
        "BCOM YoY (daily, divided by 8 for comparable scale)": bcom_yoy,
        "AU CPI YoY (quarterly, SA)": cpi_daily,
    })[start_period:]

    print(f"BCOM YoY:    {bcom_yoy.index[0]} to {bcom_yoy.index[-1]}  n={len(bcom_yoy)}")
    print(f"AU CPI YoY:  {cpi_daily.index[0]} to {cpi_daily.index[-1]}  n={len(au_cpi_yoy)}")

    runs = _above_threshold_runs(au_cpi_yoy, 3.0, start=start_period)
    axvspan = [
        {
            "xmin": pd.Period(s.to_timestamp(how="start"), freq="D"),
            "xmax": pd.Period(e.to_timestamp(how="end").normalize(), freq="D"),
            "color": "goldenrod",
            "alpha": 0.18,
        }
        for s, e in runs
    ]
    print(f"CPI > 3% runs: {len(runs)}")

    mg.line_plot_finalise(
        df.dropna(how="all"),
        title="Bloomberg Commodity Index vs Australian CPI: Year-on-Year",
        ylabel="Per cent",
        xlabel=None,
        legend={"loc": "upper left", "fontsize": "x-small"},
        annotate=True,
        rounding=1,
        width=[1.2, 1.6],
        color=["steelblue", "crimson"],
        dropna=True,
        y0=True,
        axvspan=axvspan,
        lfooter=(
            f"BCOM: daily 252-trading-day YoY change, divided by 8 for scale. "
            f"AU CPI: quarterly observations (ABS 6401.0) plotted at quarter-end days, "
            f"line connects observations only. Goldenrod: AU CPI YoY > 3%."
        ),
        rfooter="Source: Yahoo Finance, ABS",
        show=SHOW,
    )


run_bcom_vs_cpi()

BCOM YoY:    1992-01-06 to 2026-05-27  n=8636
AU CPI YoY:  1972-09-30 to 2026-03-31  n=215
CPI > 3% runs: 12


## Precious and base metals

In [18]:
METALS = [
    ("GC=F", "Gold", "USD per troy ounce"),
    ("SI=F", "Silver", "USD per troy ounce"),
    ("PL=F", "Platinum", "USD per troy ounce"),
    ("PA=F", "Palladium", "USD per troy ounce"),
    ("HG=F", "Copper", "USD per pound"),
    ("TIO=F", "Iron Ore", "USD per tonne"),
    ("MTF=F", "Coal (API2 Rotterdam)", "USD per tonne"),
]

chart_single_tickers(METALS, start=LONG_START)

Gold: 2024-03-20 to 2026-05-27  min=2157.90  max=5318.40


Silver: 2024-03-20 to 2026-05-27  min=24.48  max=115.08


Platinum: 2024-03-20 to 2026-05-27  min=894.00  max=2852.40


Palladium: 2024-03-20 to 2026-05-27  min=822.80  max=2169.90


Copper: 2024-03-20 to 2026-05-27  min=3.94  max=6.64


Iron Ore: 2024-03-20 to 2026-05-26  min=91.28  max=119.56


Coal (API2 Rotterdam): 2024-03-20 to 2025-12-26  min=89.25  max=122.90


## Natural gas benchmarks

In [19]:
def build_gas_prices() -> pd.DataFrame:
    """Fetch gas benchmarks and convert TTF to USD/MMBtu."""
    gas = fetch_yahoo_frame(
        {
            "NG=F": "Henry Hub (US)",
            "TTF=F": "TTF (Europe)",
            "JKM=F": "JKM (Asia)",
        },
        start=SHORT_START,
    ).dropna()
    eurusd = fetch_yahoo_close("EURUSD=X", start=SHORT_START)
    # Convert TTF from EUR/MWh to USD/MMBtu
    ttf_aligned = eurusd.reindex(gas.index, method="ffill")
    gas["TTF (Europe)"] = gas["TTF (Europe)"] * ttf_aligned / MMBTU_PER_MWH
    return gas


def run_natural_gas() -> None:
    """Fetch and chart natural gas benchmarks."""
    gas = build_gas_prices()
    summarise(gas, "Natural gas")
    mg.line_plot_finalise(
        gas,
        title="Natural Gas Futures Benchmarks",
        ylabel="USD per MMBtu",
        xlabel=None,
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        rounding=2,
        axvline=HORMUZ_CLOSURE,
        lfooter=f"Front-month futures. TTF converted from EUR/MWh using daily EUR/USD rate. Data to {latest_date_str(gas)}.",
        rfooter=SOURCE_YAHOO,
        show=SHOW,
    )


run_natural_gas()

Natural gas: 2025-01-02 to 2026-05-26
  Henry Hub (US)          n=350  min=2.52  max=7.46
  TTF (Europe)            n=350  min=9.07  max=20.78
  JKM (Asia)              n=350  min=9.45  max=22.35


## Grains

In [20]:
def run_grains() -> None:
    """Fetch and chart wheat and corn futures prices."""
    grains = fetch_yahoo_frame(
        {"ZW=F": "Wheat", "ZC=F": "Corn"},
        start=LONG_START,
    ).dropna()
    summarise(grains, "Grains")
    plot_frame(
        grains,
        title="Grain Futures: Wheat and Corn",
        ylabel="USD cents per bushel",
        lfooter=f"CBOT front-month futures. Data to {latest_date_str(grains)}.",
    )


run_grains()

Grains: 2024-03-20 to 2026-05-27
  Wheat                   n=549  min=495.00  max=700.25
  Corn                    n=549  min=362.00  max=502.00


## Softs and other agricultural commodities

In [21]:
SOFTS = [
    ("ZS=F", "Soybeans", "USD cents per bushel"),
    ("KC=F", "Coffee", "USD cents per pound"),
    ("SB=F", "Sugar", "USD cents per pound"),
    ("CC=F", "Cocoa", "USD per tonne"),
    ("CT=F", "Cotton", "USD cents per pound"),
    ("UFV=F", "Urea", "USD per tonne"),
]

chart_single_tickers(SOFTS, start=LONG_START)

Soybeans: 2024-03-20 to 2026-05-27  min=938.75  max=1248.00


Coffee: 2024-03-20 to 2026-05-27  min=182.40  max=438.90


Sugar: 2024-03-20 to 2026-05-27  min=13.31  max=23.42


Cocoa: 2024-03-20 to 2026-05-27  min=2798.00  max=12565.00


Cotton: 2024-03-20 to 2026-05-27  min=61.06  max=93.41


Urea: 2024-03-20 to 2026-05-26  min=283.00  max=720.25


## ASX indices

In [22]:
def run_asx_indices() -> None:
    """Fetch and chart ASX All Ordinaries and S&P/ASX 200."""
    asx = fetch_yahoo_frame(
        {"^AORD": "All Ordinaries", "^AXJO": "S&P/ASX 200"},
        start=LONG_START,
    )
    summarise(asx, "ASX")
    plot_frame(
        asx,
        title="ASX Indices: All Ordinaries and S&P/ASX 200",
        ylabel="Index",
        lfooter=f"Daily close. Data to {latest_date_str(asx)}.",
    )


run_asx_indices()
chart_single_tickers(
    [("^AXSO", "S&P/ASX Small Ordinaries", "Index")],
    start=LONG_START,
    is_futures=False,
)

ASX: 2024-03-20 to 2026-05-28
  All Ordinaries          n=554  min=7524.30  max=9435.60
  S&P/ASX 200             n=554  min=7343.30  max=9198.60


S&P/ASX Small Ordinaries: 2024-03-20 to 2026-05-28  min=2751.40  max=3992.50


## Watermark

In [23]:
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-05-28 11:21:06

Python implementation: CPython
Python version       : 3.14.2
IPython version      : 9.13.0

conda environment: n/a

Compiler    : Clang 21.1.4 
OS          : Darwin
Release     : 25.5.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

certifi  : 2026.5.20
curl_cffi: 0.15.0
mgplot   : 0.2.23
pandas   : 3.0.3
pathlib  : 1.0.1
requests : 2.34.2
typing   : 3.10.0.0
yfinance : 1.3.0

Watermark: 2.6.0

